# 04 — Classification aggregation and verification

This notebook analyzes completed HPC result shards. Split creation and cohort checks live separately in `03_split_verification.ipynb`, preventing accidental split regeneration during reporting.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

from flowlot.evaluation.repeated_benchmark import aggregate_results, load_jobs, load_registry
from flowlot.evaluation.reporting import plot_classification_suite

In [ ]:
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
RESULTS = PROJECT_ROOT / 'data' / 'results' / 'repeated_classification'
REGISTRY = RESULTS / 'shared_splits.json'
JOBS = RESULTS / 'jobs.tsv'
SHARDS = RESULTS / 'shards'
AGGREGATE = RESULTS / 'aggregate'
RUN_AGGREGATION = True
BOOTSTRAP_ITERATIONS, CONFIDENCE_LEVEL, BOOTSTRAP_SEED = 2000, 0.95, 42

## Verify HPC completion before analysis

A complete analysis requires exactly one shard for every job and one common registry checksum. Missing jobs are listed rather than silently ignored.

In [ ]:
registry = load_registry(REGISTRY)
jobs = load_jobs(JOBS)
expected = {job['job_id'] for job in jobs}
shard_paths = sorted(SHARDS.glob('*.json'))
shards = [json.loads(path.read_text()) for path in shard_paths]
observed = {shard['job_id'] for shard in shards}
wrong_registry = [shard['job_id'] for shard in shards if shard['registry_hash'] != registry['registry_hash']]
completion = pd.DataFrame([{
    'expected': len(expected), 'completed': len(expected & observed),
    'missing': len(expected - observed), 'unexpected': len(observed - expected),
    'wrong_registry': len(wrong_registry),
}])
display(completion)
if expected - observed:
    display(pd.DataFrame({'missing_job_id': sorted(expected - observed)}).head(50))
assert observed == expected and not wrong_registry, 'Complete/fix the HPC array before final analysis'

## Aggregate with patient-bootstrap confidence intervals

Repeated test predictions are averaged per patient before stratified resampling, so each patient remains one bootstrap unit.

In [ ]:
if RUN_AGGREGATION:
    integrity = aggregate_results(
        REGISTRY, JOBS, SHARDS, AGGREGATE, bootstrap_iterations=BOOTSTRAP_ITERATIONS,
        confidence_level=CONFIDENCE_LEVEL, bootstrap_seed=BOOTSTRAP_SEED,
    )
    print(json.dumps(integrity, indent=2))
else:
    print('Using existing aggregate outputs.')

## Rankings by dataset and training size

The ordinary repeated-run summary and patient-level bootstrap estimates answer different questions; both are retained.

In [ ]:
per_run = pd.read_csv(AGGREGATE / 'per_run.csv')
summary = pd.read_csv(AGGREGATE / 'summary.csv')
bootstrap = pd.read_csv(AGGREGATE / 'bootstrap_ci.csv')
paired = pd.read_csv(AGGREGATE / 'paired_comparisons.csv')
bootstrap['method'] = bootstrap[['model', 'aggregation', 'tube']].agg(' / '.join, axis=1)
ranking = bootstrap.sort_values(
    ['dataset', 'k', 'balanced_accuracy_estimate'], ascending=[True, True, False]
).groupby(['dataset', 'k']).head(10)
display(ranking[['dataset', 'k', 'method', 'n_unique_test_patients',
                 'balanced_accuracy_estimate', 'balanced_accuracy_ci_lower',
                 'balanced_accuracy_ci_upper', 'macro_f1_estimate']])
display(paired.sort_values('mean_delta_a_minus_b', ascending=False).head(30))

In [ ]:
display(Image(filename=AGGREGATE / 'aggregation_comparison.png'))
figure, axis = plt.subplots(figsize=(7, 4))
best_at_max_k = ranking[ranking['k'] == ranking['k'].max()].head(15)
errors = np.vstack([
    best_at_max_k['balanced_accuracy_estimate'] - best_at_max_k['balanced_accuracy_ci_lower'],
    best_at_max_k['balanced_accuracy_ci_upper'] - best_at_max_k['balanced_accuracy_estimate'],
])
axis.errorbar(best_at_max_k['balanced_accuracy_estimate'], range(len(best_at_max_k)), xerr=errors, fmt='o')
axis.set_yticks(range(len(best_at_max_k)), best_at_max_k['method'])
axis.set(xlabel='Balanced accuracy (bootstrap CI)', ylabel='Method', xlim=(0, 1.02))
sns.despine()
plt.show()

## Best-method patient-level diagnostics

Select a method/k below. Predictions are averaged across repeated test appearances before the confusion matrix, ROC, and PR figures are generated.

In [ ]:
selected = ranking.iloc[0]
selected_shards = [shard for shard in shards if (
    shard['model'] == selected['model'] and shard['aggregation'] == selected['aggregation']
    and shard['tube'] == selected['tube'] and shard['k'] == selected['k']
)]
patient_predictions, patient_labels = {}, {}
for shard in selected_shards:
    for patient, label, probability in zip(shard['test_ids'], shard['y_true'], shard['probabilities']):
        patient_labels[patient] = label
        patient_predictions.setdefault(patient, []).append(probability)
patient_ids = sorted(patient_predictions)
y_true = np.asarray([patient_labels[patient] for patient in patient_ids])
probabilities = np.stack([np.mean(patient_predictions[patient], axis=0) for patient in patient_ids])
diagnostic_dir = AGGREGATE / 'best_method_diagnostics'
plot_classification_suite(y_true, probabilities, diagnostic_dir, registry['class_names'], random_state=BOOTSTRAP_SEED)
patient_table = pd.DataFrame({
    'patient_id': patient_ids, 'true': y_true, 'predicted': probabilities.argmax(1),
    'confidence': probabilities.max(1),
}).sort_values(['true', 'confidence'])
display(patient_table)
print('Diagnostic figures:', diagnostic_dir.resolve())